# Module 5 — Rebalancing & Glide Path

## Two ideas

**Glide Path:** Your optimal allocation at age 28 is NOT optimal at age 55. As a goal date nears, you can't afford a 40% drawdown — you don't have time to recover. Real target-date funds (Vanguard Target Retirement, HDFC Retirement Fund) automatically shift from equity-heavy → debt-heavy as the date approaches. We replicate that logic here.

**Drift Detection:** Markets don't move equally. After a 3-year bull run, your 60% equity allocation might be sitting at 72%. You're now taking more risk than you signed up for — and the next crash hits harder. Rebalancing restores the target and forces you to *sell high, buy low* systematically.

In [ ]:
# ── Setup (Colab) ────────────────────────────────────────────────
!pip install -q numpy pandas plotly
import os
if not os.path.exists('robo-advisor'):
    !git clone https://github.com/parinmody30/robo-advisor.git
else:
    !git -C robo-advisor pull

import sys
sys.path.insert(0, 'robo-advisor/src')
DATA_DIR = 'robo-advisor/data'
print('Setup complete.')

In [ ]:
import json
import pandas as pd
import plotly.graph_objects as go
from optimizer import ASSET_LABELS, ASSET_COLORS
from rebalancer import compute_glide_path, detect_drift, simulate_drift, rebalance_summary

with open(f'{DATA_DIR}/allocation.json') as f:
    allocation = json.load(f)
with open(f'{DATA_DIR}/goals.json') as f:
    goals_data = json.load(f)

weights  = allocation['weights']
persona  = allocation['persona']
horizon  = goals_data['goals'][0]['years_to_goal']
print(f'Persona: {persona} | Horizon: {horizon} years')

## 5A. Glide path — equity vs debt over time

In [ ]:
glide = compute_glide_path(weights, horizon_years=horizon)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=glide.years, y=glide.equity_pct,
    name='Equity %', mode='lines+markers',
    line=dict(color='#1565C0', width=3),
    fill='tozeroy', fillcolor='rgba(21,101,192,0.15)',
))
fig.add_trace(go.Scatter(
    x=glide.years, y=glide.debt_pct,
    name='Debt + Fixed %', mode='lines+markers',
    line=dict(color='#2E7D32', width=3),
    fill='tozeroy', fillcolor='rgba(46,125,50,0.15)',
))
fig.update_layout(
    title='Glide Path — Portfolio Shifts from Growth to Safety as Goal Approaches',
    xaxis_title='Years from now (0 = goal date)',
    yaxis_title='Allocation %',
    template='plotly_white', height=460,
)
fig.show()

## 5B. Annual allocation heatmap across the glide path

In [ ]:
# Show every 5th year for readability
step = max(1, horizon // 8)
selected_years = [y for y in glide.years if y % step == 0 or y == horizon]
selected_weights = [glide.annual_weights[y] for y in selected_years]

assets = list(weights.keys())
labels = [ASSET_LABELS.get(a, a) for a in assets]
matrix = [[w.get(a, 0)*100 for a in assets] for w in selected_weights]

fig2 = go.Figure(go.Heatmap(
    z=matrix,
    x=labels,
    y=[f'Year {y}' for y in selected_years],
    colorscale='Blues',
    text=[[f'{v:.1f}%' for v in row] for row in matrix],
    texttemplate='%{text}',
))
fig2.update_layout(
    title='Allocation % by Asset Across Glide Path',
    height=max(350, len(selected_years)*45),
    template='plotly_white',
)
fig2.show()

## 5C. Drift detection
Simulate what happens after a 3-year bull market and check if rebalancing is needed.

In [ ]:
# Simulate a 3-year equity bull run: equity assets return ~18%, debt ~7%
bull_returns = {
    'Equity_LargeCap': 0.18, 'Equity_MidCap': 0.22, 'Equity_SmallCap': 0.25,
    'Equity_Intl': 0.15, 'ELSS': 0.20,
    'Gold': 0.08, 'Silver': 0.10,
    'Debt_Gilt': 0.07, 'Debt_Corporate': 0.08,
    'FixedDeposit': 0.07, 'PPF': 0.071, 'RBI_Bond': 0.0805, 'LiquidFund': 0.065,
}

drifted = simulate_drift(weights, bull_returns, years=3)

print('Weight comparison after 3-year bull run:')
print(f'{"Asset":<25} {"Target":>8} {"After 3yr":>10} {"Drift":>8}')
print('-' * 55)
for a in weights:
    t = weights.get(a, 0)
    d = drifted.get(a, 0)
    diff = d - t
    flag = ' ⚠️' if abs(diff) >= 0.05 else ''
    print(f'{ASSET_LABELS.get(a,a):<25} {t*100:>7.1f}%  {d*100:>8.1f}%  {diff*100:>+7.1f}%{flag}')

## 5D. Rebalancing instructions

In [ ]:
# Assume ₹10 lakh portfolio value
portfolio_value = 1_000_000

actions = detect_drift(weights, drifted, portfolio_value, threshold=0.05)
df = rebalance_summary(actions)

if df.empty:
    print('✅ No rebalancing needed — all assets within ±5% of target.')
else:
    print(f'Rebalancing required for ₹{portfolio_value:,} portfolio:')
    print(df.to_string(index=False))

## 5E. Visualise drift — before vs after

In [ ]:
assets   = list(weights.keys())
labels_a = [ASSET_LABELS.get(a, a) for a in assets]
target_w = [weights.get(a, 0)*100 for a in assets]
drifted_w= [drifted.get(a, 0)*100 for a in assets]

fig3 = go.Figure()
fig3.add_trace(go.Bar(name='Target', x=labels_a, y=target_w,
                      marker_color='#1565C0', opacity=0.85))
fig3.add_trace(go.Bar(name='After 3yr bull run', x=labels_a, y=drifted_w,
                      marker_color='#E53935', opacity=0.85))
fig3.update_layout(
    barmode='group',
    title='Portfolio Drift After 3-Year Bull Market',
    xaxis_tickangle=-35,
    yaxis_title='Weight %',
    template='plotly_white', height=480,
)
fig3.show()

## Theory note

**Why rebalance at all?** Two reasons:
1. **Risk control** — drift means you're unknowingly taking more risk than your profile allows
2. **Systematic buy low / sell high** — rebalancing forces you to sell what has appreciated and buy what has lagged, which is mechanically contrarian

**When to rebalance — threshold vs calendar:**
| Approach | Trigger | Pros | Cons |
|---|---|---|---|
| Calendar (annual) | Every 12 months | Simple, predictable | May miss large drifts |
| Threshold (±5%) | When any asset drifts >5% | Responds to actual drift | More transactions, tax events |
| Hybrid | Calendar check + threshold | Best of both | Slightly more complex |

We use **threshold-based** (±5%) as the default — it's more responsive and avoids unnecessary trades in stable markets.

**Glide path caveat:** The linear interpolation here is a simplification. Real target-date funds use actuarial tables and liability-matching. For an individual with a single goal, linear is a good approximation.